# DataCollector: social-model integration

Select the environment installed from `checks/requirements.txt` and use **Run All**. This notebook covers point 1 of the [validation sequence](checks/README.md): the three social rules, synchronous updates, reflecting walls and the DataCollector run together before adding light.

The shared implementation is [shiner_model.py](shiner_model.py). Keeping it in one file means this notebook and the five notebooks in `checks/` test the same code. It is a deterministic, constant-speed baseline with 360° perception and instantaneous turns.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "shiner_model.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
from shiner_model import GoldenShiners, GoldenShinersModel, compute_group_metrics, reflect
from checks.check_support import controlled_model, audit_run


## What was simplified and corrected

- `decide()` plans the next position and heading; `advance()` commits them. Keeping these two short phases makes every fish observe the same old state. Combining them into sequential per-fish updates would change the model.
- Constant speed needs no separate `next_speed` variable. Reflection operates on the proposed state and supports multiple wall crossings.
- Exact overlaps now receive deterministic opposite escape directions. Different headings supply the difference vector; identical headings use opposite perpendiculars, with agent ID breaking the tie. Positions are not teleported and the random generator is not advanced. This is a numerical convention, not a measured fish interaction.
- Repulsion remains active whenever a close neighbour exists. If the repulsion sum cancels, the fish escapes from the nearest neighbour, using ID only to break equal-distance ties.
- Orientation and attraction are combined as **raw vector sums**, then normalized once, matching the social-vector equations in [Couzin et al. (2002)](https://doi.org/10.1006/jtbi.2002.3065) and the [Puckett et al. (2018) supplement](https://static-content.springer.com/esm/art%3A10.1038%2Fs41598-018-26037-9/MediaObjects/41598_2018_26037_MOESM1_ESM.pdf). Previously each zone was normalized separately. A cancelling combined social vector now retains the current heading.
- Repeated reporter lambdas are generated from the metric names. Parameters are validated when creating the model. DataCollector records the initial state and then observes each completed step.

These corrections change some trajectories relative to the earlier equal-zone-weighted version. That is expected; the five check notebooks use this cleaned implementation consistently.


In [2]:
import inspect

# Display the actual shared implementation, rather than a second class definition.
print(inspect.getsource(GoldenShiners.decide))
print(inspect.getsource(GoldenShiners.advance))

    def decide(self):
        """Plan without mutating any current position, heading or speed."""
        direction = self.social_direction(self.get_neighbors())
        position = self.position + direction * self.speed * self.model.dt
        self.next_position, self.next_direction = reflect(position, direction, self.model.bounds)

    def advance(self):
        """Commit the state computed while all fish still had their old state."""
        self.position = self.next_position.copy()
        self.direction = self.next_direction.copy()



In [3]:
model = GoldenShinersModel(width=20, height=20, n_fish=20, speed=1.0, seed=1)
for _ in range(100):
    model.step()
model_data = model.datacollector.get_model_vars_dataframe()
agent_data = model.datacollector.get_agent_vars_dataframe()
pd.DataFrame([audit_run(model)])

,Fish,Steps,Model rows,Agent rows,Result
0,20,100,101,2020,passed


In [4]:
model_data

,Polarization,Rotation,MeanSpeed,MeanNND,MeanDistanceToCentroid,CentroidX,CentroidY,CollectiveState
0,0.296489,0.055420,1.0,3.112169,7.821139,9.306132,9.619529,swarm
1,0.204176,0.010182,1.0,3.018089,7.798115,9.292531,9.597934,swarm
2,0.158882,0.002917,1.0,2.965809,7.763717,9.276946,9.585624,swarm
3,0.044727,0.018960,1.0,2.868665,7.719782,9.272999,9.581664,swarm
4,0.055220,0.049136,1.0,2.806850,7.670806,9.270731,9.575145,swarm
...,...,...,...,...,...,...,...,...
96,0.997294,0.030356,1.0,0.605056,1.259101,11.298342,12.923309,polarized
97,0.998326,0.023646,1.0,0.600970,1.256248,11.344404,13.039288,polarized
98,0.898804,0.121005,1.0,0.604755,1.262464,11.384398,13.144279,polarized
99,0.797936,0.025092,1.0,0.625053,1.266786,11.420067,13.237425,polarized


In [5]:
agent_data

X          Y      DirX      DirY  Speed
Step AgentID                                                 
0    1        10.236432  19.009274  0.617071  0.786908    1.0
     2        18.972989   6.236629 -0.886184  0.463334    1.0
     3        16.554052   8.183983 -0.951842 -0.306588    1.0
     4         0.551182  15.070262 -0.971418 -0.237374    1.0
     5         6.594634  15.768574 -0.328045  0.944662    1.0
...                 ...        ...       ...       ...    ...
100  16       11.470430  13.481458  0.357778  0.933806    1.0
     17       10.595902  14.438586  0.363785  0.931483    1.0
     18       12.338079  12.728201  0.357778  0.933806    1.0
     19       10.182606  11.391214  0.483137  0.875545    1.0
     20       12.559169  14.944718  0.201899  0.979406    1.0

[2020 rows x 5 columns]

## Controlled integration cases

Random runs alone could miss a branch. These hand-built configurations exercise each social rule, unequal zone populations, rule priority, exact and near overlaps, zone boundaries, cancelling vectors, synchronous updates and wall reflections. The expected headings come from simple geometry. Comparing saved positions after the synchronous case also checks that collection happens after movement.


In [6]:
# Each expected heading is chosen from the geometry, independently of the code.
rule_cases = [
    ("No neighbours", [(10, 10), (16, 10)], [(1, 0), (0, 1)], (1, 0)),
    ("Repulsion", [(10, 10), (10.25, 10)], [(0, 1), (0, 1)], (-1, 0)),
    ("Orientation", [(10, 10), (11, 10)], [(1, 0), (0, 1)], (0, 1)),
    ("Attraction", [(10, 10), (14, 10)], [(0, 1), (-1, 0)], (1, 0)),
    ("Orientation + attraction", [(10, 10), (11, 10), (14, 10)],
     [(-1, 0), (0, 1), (-1, 0)], (1 / np.sqrt(2), 1 / np.sqrt(2))),
    ("Raw sums: two headings + one attractor", [(10, 10), (11, 10), (10, 11), (14, 10)],
     [(-1, 0), (0, 1), (0, 1), (1, 0)], (1 / np.sqrt(5), 2 / np.sqrt(5))),
    ("Repulsion priority", [(10, 10), (10.25, 10), (11, 10), (14, 10)],
     [(0, 1), (0, 1), (0, 1), (0, 1)], (-1, 0)),
    ("Cancelled repulsion still has priority", [(10, 10), (9.75, 10), (10.25, 10), (10, 14)],
     [(0, 1), (1, 0), (1, 0), (1, 0)], (1, 0)),
    ("Boundary 0.5: orientation", [(10, 10), (10.5, 10)], [(-1, 0), (0, 1)], (0, 1)),
    ("Boundary 3.0: attraction", [(10, 10), (13, 10)], [(-1, 0), (0, 1)], (1, 0)),
    ("Boundary 5.5: attraction", [(10, 10), (15.5, 10)], [(-1, 0), (0, 1)], (1, 0)),
    ("Outside 5.5: no interaction", [(10, 10), (15.500001, 10)], [(-1, 0), (0, 1)], (-1, 0)),
    ("Near overlap remains finite", [(10, 10), (10.000000001, 10)], [(0, 1), (0, 1)], (-1, 0)),
    ("Cancelled social sum keeps current heading", [(10, 10), (11, 10), (14, 10)],
     [(0, 1), (-1, 0), (0, 1)], (0, 1)),
]

rule_results = []
for name, positions, headings, expected in rule_cases:
    test_model = controlled_model(positions, headings)
    before_positions = np.array([f.position.copy() for f in test_model.agents])
    before_headings = np.array([f.direction.copy() for f in test_model.agents])
    test_model.agents.do("decide")
    np.testing.assert_allclose(next(iter(test_model.agents)).next_direction, expected, rtol=0, atol=1e-12)
    np.testing.assert_array_equal([f.position for f in test_model.agents], before_positions)
    np.testing.assert_array_equal([f.direction for f in test_model.agents], before_headings)
    rule_results.append({"Check": name, "Result": "passed"})

# Identical positions AND identical headings used to leave this pair overlapped.
test_model = controlled_model([(10, 10), (10, 10)], [(1, 0), (1, 0)])
left, right = test_model.agents
np.testing.assert_allclose(left.overlap_direction(right), -right.overlap_direction(left))
test_model.step()
separation = np.linalg.norm(left.position - right.position)
np.testing.assert_allclose(separation, 0.25, rtol=0, atol=1e-12)
assert np.isfinite([left.direction, right.direction]).all()
rule_results.append({"Check": "Identical overlap separates without random kicks", "Result": "passed"})

# Even almost-identical headings must give exactly opposite pairwise escapes.
test_model = controlled_model(
    [(10, 10), (10, 10)], [(1, 0), (np.cos(1e-13), np.sin(1e-13))]
)
left, right = test_model.agents
np.testing.assert_array_equal(left.overlap_direction(right), -right.overlap_direction(left))
rule_results.append({"Check": "Near-identical overlapping headings remain antisymmetric", "Result": "passed"})

# A larger coincident group must also recover a finite, separated configuration.
test_model = controlled_model([(10, 10)] * 5, [(1, 0)] * 5)
for _ in range(20):
    test_model.step()
positions = np.array([f.position for f in test_model.agents])
distances = np.linalg.norm(positions[:, None] - positions[None, :], axis=2)
np.fill_diagonal(distances, np.inf)
assert distances.min() > 0
assert audit_run(test_model)["Result"] == "passed"
rule_results.append({"Check": "Five-fish coincident start recovers", "Result": "passed"})

# Using old headings should swap these headings; sequential motion would fail.
test_model = controlled_model([(10, 10), (11, 10)], [(1, 0), (0, 1)])
test_model.step()
np.testing.assert_allclose([f.direction for f in test_model.agents], [(0, 1), (1, 0)])
np.testing.assert_allclose([f.position for f in test_model.agents], [(10, 10.125), (11.125, 10)])
np.testing.assert_allclose(
    test_model.datacollector.get_agent_vars_dataframe().xs(1, level="Step")[["X", "Y"]],
    [(10, 10.125), (11.125, 10)],
)
rule_results.append({"Check": "Synchronous decisions and recording after advance", "Result": "passed"})

# A wall changes only the planned state until advance commits it.
test_model = controlled_model([(19.95, 10)], [(1, 0)])
test_fish = next(iter(test_model.agents))
test_fish.decide()
np.testing.assert_allclose(test_fish.position, (19.95, 10))
np.testing.assert_allclose(test_fish.direction, (1, 0))
np.testing.assert_allclose(test_fish.next_position, (19.925, 10))
np.testing.assert_allclose(test_fish.next_direction, (-1, 0))
test_model.step()
np.testing.assert_allclose(test_fish.position, (19.925, 10))
rule_results.append({"Check": "Wall reflection preserves the old state", "Result": "passed"})

for proposed_x, expected_x, expected_dx in [(41, 1, 1), (61, 19, -1), (-41, 1, 1)]:
    direction = np.array([1.0 if proposed_x > 0 else -1.0, 0.0])
    position, reflected = reflect(np.array([proposed_x, 10.0]), direction, np.array([[0, 20], [0, 20]]))
    np.testing.assert_allclose(position, (expected_x, 10))
    np.testing.assert_allclose(reflected, (expected_dx, 0))
rule_results.append({"Check": "Multiple wall crossings are reflected correctly", "Result": "passed"})

for invalid_count in (0, -1):
    try:
        GoldenShinersModel(n_fish=invalid_count)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid fish count accepted")
rule_results.append({"Check": "Invalid fish counts rejected", "Result": "passed"})
rule_results = pd.DataFrame(rule_results)
print(f"{len(rule_results)} controlled integration checks passed.")
rule_results


21 controlled integration checks passed.


,Check,Result
0,No neighbours,passed
1,Repulsion,passed
2,Orientation,passed
3,Attraction,passed
4,Orientation + attraction,passed
5,Raw sums: two headings + one attractor,passed
6,Repulsion priority,passed
7,Cancelled repulsion still has priority,passed
8,Boundary 0.5: orientation,passed
9,Boundary 3.0: attraction,passed


## Point 1 — checks and interpretation

The executed cells above verify the social model and DataCollector together:

1. **Three local rules:** separate orientation, attraction and repulsion cases produce the expected headings. Unequal neighbour counts distinguish raw-sum combination from the previous equal-zone weighting. Repulsion wins even when its vector sum cancels.
2. **Defined overlap behaviour:** two initially coincident, identically headed fish separate by 0.25 body lengths after one step at speed 1 and `dt=0.125`. A five-fish coincident start also recovers finite separated positions. These checks target the previous `distance == 0: continue` failure.
3. **Synchronous updates:** planning leaves current positions and headings unchanged. The two-fish heading exchange gives the expected result only when all decisions use the old state. DataCollector records the committed positions.
4. **Numerical integrity:** 100 updates produce 101 group records and 2020 individual records for 20 fish, with finite values, unit headings and positions inside the arena. A single fish intentionally has undefined (`NaN`) nearest-neighbour distance.
5. **Boundary handling:** reflections do not mutate the old state and can handle a proposed movement crossing multiple walls. This checks containment, not collision avoidance between moving fish.

Passing these cases establishes integration for the tested baseline. Whether collective motion is robust across seeds is examined in [check 04](checks/04_schooling_across_seeds.ipynb), using continuous spacing and order metrics rather than one label.

Before reproducing the Puckett model, perception, a turn limit and social noise still require implementation and validation. Its supplement lists 1.75 radians **per update** (about 100°), not 100° per second. Light-dependent speed is still absent. At the proposed maximum speed of 5 BL/s, one update would cover 0.625 BL, greater than the 0.5-BL repulsion radius, so speed/timestep and crossing tests are required before adding that extension.

## Remaining checks

- [02 — initial state and recording integrity](checks/02_recording_integrity.ipynb)
- [03 — known-answer metric validation](checks/03_metric_validation.ipynb)
- [04 — schooling across seeds and ablations](checks/04_schooling_across_seeds.ipynb)
- [05 — collective-state labels as summaries](checks/05_collective_state_summary.ipynb)
- [06 — diagnostic plots before PyVista](checks/06_diagnostic_plots.ipynb)

Order parameters and the exploratory 0.35/0.65 thresholds follow [Tunstrøm et al. (2013)](https://doi.org/10.1371/journal.pcbi.1002915). These notebooks use raw per-update measurements; that experiment used temporal smoothing. The labels do not themselves establish cohesion or biological agreement.


## Follow-up validation before light

The [pre-light handling history](handlings/README.md) preserves three subsequent trials, their saved results and the remaining timestep problem. The [next proposal](handlings/NEXT_SOLUTION.md) separates social decisions from physics updates and has not been implemented.